# Streaming ML Framework — Demo Notebook

## Framework 簡介

本 Framework 是一套**純 NumPy 實作**的串流機器學習框架，專為資料以「分批（chunk）」形式陸續抵達的場景設計。
框架完全不依賴 scikit-learn、scipy 等第三方 ML 函式庫，僅使用 `numpy` 與 `matplotlib`。

| 模組 | 功能 |
|---|---|
| `io.py` | 自訂 CSV 讀寫、串流生成器、資料切分 |
| `preprocessing.py` | `StandardScaler`、`MinMaxScaler`、`Imputer`，全部支援 `partial_fit` |
| `stats.py` | 串流統計（`StreamStats`、`chunk_mean/variance/quantile/histogram`） |
| `tree.py` | 決策樹分類器（Gini / Entropy，支援 `partial_fit` 增量學習） |
| `ensemble.py` | `EnsembleClassifier`（Bagging / Random Forest），支援串流 |
| `metrics.py` | 串流指標（`Accuracy`、`F1Score`、`ConfusionMatrix`）及批次版函數 |
| `pipeline.py` | `Pipeline`：將 transformer + estimator 串聯，支援 `partial_fit` |
| `stream.py` | `StreamTrainer`：自動迭代 chunk、記錄指標、追蹤記憶體 |
| `visualise.py` | 繪圖工具（指標趨勢、模型比較、預測散點、混淆矩陣） |

### 本 Demo 涵蓋的四個核心要求

1. **使用 `io.py` 從 CSV 載入資料集**
2. **將資料集分割成多個 chunk，模擬串流資料情境**
3. **對每個 chunk 呼叫 `.partial_fit()` 進行增量訓練**
4. **使用 `visualise.py` 記錄並視覺化關鍵指標（accuracy、error rate、模型比較）**

In [ ]:
# Cell 1 — 載入套件、產生合成資料集並以 io.save_csv 存檔，再以 io.load_csv 讀回（核心要求 1）
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt

from framework.io import load_csv, save_csv, split_into_chunks
from framework.preprocessing import StandardScaler
from framework.ensemble import RandomForestClassifier, EnsembleClassifier
from framework.pipeline import Pipeline
from framework.stream import StreamTrainer
from framework.metrics import Accuracy, F1Score, confusion_matrix as compute_cm
from framework.visualise import (
    plot_metric_over_time, compare_models,
    plot_predictions_vs_ground_truth, plot_confusion_matrix,
)

# 產生合成資料集（1000 筆 × 6 特徵，二元分類）
rng = np.random.default_rng(seed=0)
N, D = 1000, 6
X_raw = rng.normal(loc=0.0, scale=2.0, size=(N, D))
weights = np.array([1.5, -1.0, 0.8, 0.0, 0.0, 0.0])
y_raw = (X_raw @ weights > 0).astype(int)

# 核心要求 1：以 io.save_csv 寫入 CSV，再以 io.load_csv 讀回
csv_path = '/tmp/demo_data.csv'
headers = [f'f{i}' for i in range(D)] + ['label']
save_csv(csv_path, np.column_stack([X_raw, y_raw]), headers=headers)

data, cols = load_csv(csv_path, has_header=True)
X = data[:, :-1]
y = data[:, -1].astype(int)

print(f'資料集：{X.shape}，類別分布：{np.bincount(y)}')
print(f'欄位：{cols}')

In [ ]:
# Cell 2 — 切分 chunk 模擬串流（核心要求 2）並建立兩條 Pipeline

# 核心要求 2：以 split_into_chunks 切分為 10 個 chunk
N_CHUNKS = 10
chunks = split_into_chunks(X, y, n_chunks=N_CHUNKS)
print(f'切分為 {N_CHUNKS} 個 chunk，每個 {N // N_CHUNKS} 筆')

# 建立 Random Forest 與 Bagging 兩條 Pipeline
def make_rf_pipeline(seed=42):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', RandomForestClassifier(n_estimators=10, max_depth=5, random_state=seed)),
    ])

def make_bag_pipeline(seed=42):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf', EnsembleClassifier(n_estimators=10, method='bagging', max_depth=5, random_state=seed)),
    ])

print('RF Pipeline  : StandardScaler -> RandomForestClassifier(n=10, depth=5)')
print('Bag Pipeline : StandardScaler -> EnsembleClassifier(bagging, n=10, depth=5)')

In [ ]:
# Cell 3 — 增量訓練：對每個 chunk 呼叫 .partial_fit()（核心要求 3）
# StreamTrainer 內部呼叫 pipeline.partial_fit(X_chunk, y_chunk)，
# 指標為「累積型」，反映截至目前所有資料上的整體表現。

rf_trainer = StreamTrainer(
    pipeline=make_rf_pipeline(seed=42),
    metrics=[Accuracy(), F1Score()],
    log_memory=True,
)
bag_trainer = StreamTrainer(
    pipeline=make_bag_pipeline(seed=42),
    metrics=[Accuracy(), F1Score()],
    log_memory=True,
)

print('Chunk | RF Acc  RF F1  | Bag Acc Bag F1')
print('-' * 45)
for i, (Xc, yc) in enumerate(chunks):
    rf  = rf_trainer.fit_chunk(Xc, yc)
    bag = bag_trainer.fit_chunk(Xc, yc)
    print(f'  {i:2d}  | {rf["accuracy"]:.4f}  {rf["f1score"]:.4f}  | {bag["accuracy"]:.4f}  {bag["f1score"]:.4f}')

rf_log  = rf_trainer.get_log()
bag_log = bag_trainer.get_log()
rf_acc  = [r['accuracy'] for r in rf_log]
rf_f1   = [r['f1score']  for r in rf_log]
bag_acc = [r['accuracy'] for r in bag_log]
bag_f1  = [r['f1score']  for r in bag_log]

In [ ]:
# Cell 4 — Accuracy、Error Rate、F1 趨勢圖（核心要求 4）
# 以 plot_metric_over_time 繪製 RF 模型三個指標隨 chunk 的累積變化。

rf_error = [1.0 - a for a in rf_acc]

plot_metric_over_time(
    rf_acc, title='RF — Cumulative Accuracy over Chunks',
    ylabel='Accuracy', save_path='/tmp/rf_accuracy.png',
)
plt.show()

plot_metric_over_time(
    rf_error, title='RF — Cumulative Error Rate over Chunks',
    ylabel='Error Rate (1 - Accuracy)', save_path='/tmp/rf_error.png',
)
plt.show()

plot_metric_over_time(
    rf_f1, title='RF — Cumulative F1 Score over Chunks',
    ylabel='F1 Score', save_path='/tmp/rf_f1.png',
)
plt.show()

print(f'Accuracy  : {rf_acc[0]:.4f} -> {rf_acc[-1]:.4f}')
print(f'Error Rate: {rf_error[0]:.4f} -> {rf_error[-1]:.4f}')
print(f'F1 Score  : {rf_f1[0]:.4f} -> {rf_f1[-1]:.4f}')

In [ ]:
# Cell 5 — 模型比較：RF vs Bagging（核心要求 4）
# 以 compare_models 在同一張圖疊加兩個模型的 Accuracy 與 F1 曲線。

compare_models(
    rf_acc, bag_acc,
    labels=['Random Forest', 'Bagging'],
    title='Model Comparison — Cumulative Accuracy',
    ylabel='Accuracy',
    save_path='/tmp/model_comparison_acc.png',
)
plt.show()

compare_models(
    rf_f1, bag_f1,
    labels=['Random Forest', 'Bagging'],
    title='Model Comparison — Cumulative F1 Score',
    ylabel='F1 Score',
    save_path='/tmp/model_comparison_f1.png',
)
plt.show()

print(f'RF  最終：Accuracy={rf_acc[-1]:.4f}，F1={rf_f1[-1]:.4f}')
print(f'Bag 最終：Accuracy={bag_acc[-1]:.4f}，F1={bag_f1[-1]:.4f}')

In [ ]:
# Cell 6 — 預測散點圖 & 混淆矩陣（核心要求 4）
# 以最後一個 chunk 視覺化 RF 的預測結果與混淆矩陣。

Xlast, ylast = chunks[-1]
y_pred_last = rf_trainer.pipeline.predict(Xlast)

plot_predictions_vs_ground_truth(
    ylast, y_pred_last,
    title='RF Predictions vs Ground Truth — Last Chunk',
    save_path='/tmp/pred_vs_true.png',
)
plt.show()

cm = compute_cm(ylast, y_pred_last)
plot_confusion_matrix(
    cm, class_names=['Class 0', 'Class 1'],
    save_path='/tmp/confusion_matrix.png',
)
plt.show()

n_correct = int(np.sum(y_pred_last == ylast))
print(f'最後 chunk：正確 {n_correct}/{len(ylast)}，accuracy={n_correct/len(ylast):.4f}')
print(f'混淆矩陣：TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}')

In [ ]:
# Cell 7 — 記憶體用量追蹤 & 最終摘要（核心要求 4）

mem_mb = [r.get('memory_mb', 0.0) for r in rf_log]
plot_metric_over_time(
    mem_mb,
    title='RF Pipeline — Memory Usage (RSS) over Chunks',
    ylabel='RSS Memory (MB)',
    save_path='/tmp/memory_usage.png',
)
plt.show()
print(f'記憶體：初始={mem_mb[0]:.1f} MB，峰值={max(mem_mb):.1f} MB\n')

print('=' * 48)
print('         串流訓練最終結果摘要')
print('=' * 48)
print(f'  資料集  : {N} 筆 x {D} 特徵，二元分類')
print(f'  Chunks  : {N_CHUNKS} 個，每個 {N // N_CHUNKS} 筆')
print()
header = f"  {'模型':<18} {'Accuracy':>10} {'F1 Score':>10}"
print(header)
print(f'  {"-"*40}')
print(f'  {"Random Forest":<18} {rf_acc[-1]:>10.4f} {rf_f1[-1]:>10.4f}')
print(f'  {"Bagging":<18} {bag_acc[-1]:>10.4f} {bag_f1[-1]:>10.4f}')
print()
print('  核心要求確認：')
print('  1. io.load_csv 從 CSV 載入資料集')
print('  2. split_into_chunks 切分為 10 個 chunk（模擬串流）')
print('  3. pipeline.partial_fit() 對每個 chunk 增量訓練')
print('  4. visualise.py 視覺化：Accuracy / Error / F1 / 模型比較 / 散點 / 混淆矩陣 / 記憶體')